In [2]:
import sys
print(sys.version)
print(spark.version)

3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:38:13) [GCC 12.3.0]
3.5.1


In [15]:
import os
import subprocess
import datetime
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import functions as F
from pyspark.sql.types import *

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

## Read Amazon reviews and meta data from GCS

#### ALL Reviews

In [4]:
gcs_folder = 'gs://msca-bdp-data-open/final_project_reviews'

In [5]:
%%time   
    
df_reviews = spark.read.parquet(os.path.join(gcs_folder, 'reviews_parquet'))
print(f'Records read from dataframe *reviews*: {df_reviews.count():,.0f}')

Records read from dataframe *reviews*: 64,679,785
CPU times: user 22.8 ms, sys: 5.88 ms, total: 28.7 ms
Wall time: 10.7 s


In [6]:
display(df_reviews.printSchema())

# ASIN (Amazon Standard Identification Number)
# : unique 10-character code Amazon assigns to each product for internal tracking

root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)


None

In [10]:
df_reviews.limit(5)

asin,helpful_vote,parent_asin,rating,text,timestamp,title,user_id,verified_purchase
B00HSAOFD8,0,B00HSAOFD8,4.0,Brushes arrived q...,1453819207000,Four Stars,AGWL43EG6H3AO3CN2...,true
B07MGKRMBG,0,B07SPY1GJG,5.0,I have tried just...,1600341347791,Finally a natural...,AGUUIUEN3Y4X54WXM...,true
B01FOJ7EVO,0,B09J7FM96G,4.0,Love Mediheal but...,1607280179211,Too small for me,AGOX7WJIY67A3OOKT...,true
B01B8ABHK2,0,B016O0QZEO,5.0,blended nicely wi...,1609835116809,Quality,AH26NV6K35W6YPPWJ...,true
B002W80GMK,0,B002W80GMK,5.0,Love this product,1520895879064,Five Stars,AHMBBCUUUIXNXTUAB...,true


In [7]:
spark.conf.set("spark.sql.repl.eagerEval.enabled",True)

#### ALL Meta data

In [8]:
%%time

df_meta = spark.read.parquet(os.path.join(gcs_folder, 'meta_parquet'))
print(f'Records read from dataframe *meta*: {df_meta.count():,.0f}')

Records read from dataframe *meta*: 4,320,533
CPU times: user 5.6 ms, sys: 4.47 ms, total: 10.1 ms
Wall time: 2.01 s


In [9]:
display(df_meta.printSchema())

root
 |-- author: struct (nullable = true)
 |    |-- about: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- avatar: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- bought_together: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- main_category: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- price: string (nullable = true)
 |-- rating_number: long (nullable = true)
 |-- store: string (nullable = true)
 |-- subtitle: string (nullable = true)
 |-- title: string (nullable = true)


None

In [14]:
df_meta.limit(5)

author,average_rating,bought_together,categories,description,main_category,parent_asin,price,rating_number,store,subtitle,title
NULL,5.0,NULL,[Beauty & Persona...,[],All Beauty,B0771WJBHK,NULL,5,Bare Alchemy,NULL,Soothe Serum for ...
NULL,4.0,NULL,[Beauty & Persona...,[],NULL,B09NKNNS57,NULL,17,DISCOVER DEVICE,NULL,DISCOVER Tattoo B...
NULL,4.6,NULL,[Beauty & Persona...,[],All Beauty,B0C6FF5HWL,8.99,137,auroray,NULL,auroray 2200 Pcs ...
NULL,1.0,NULL,[Beauty & Persona...,[],NULL,B09NKSHV3Z,15.69,2,KKNC,NULL,KKNC Chemistry In...
NULL,5.0,NULL,[Beauty & Persona...,[Introduced by th...,All Beauty,B000C213AC,NULL,1,Fine Fragrances,NULL,Aficionado By Fin...


## Discard irrelevant/erroneous data

#### Reviews

In [17]:
reviews_profile = df_reviews.agg(
    F.min("rating").alias("min_rating"),
    F.max("rating").alias("max_rating"),
    F.avg("rating").alias("avg_rating"),
    F.min("helpful_vote").alias("min_helpful_vote"),
    F.max("helpful_vote").alias("max_helpful_vote"),
    F.countDistinct("asin").alias("n_distinct_asin"),
    F.countDistinct("parent_asin").alias("n_distinct_parent_asin"),
    F.countDistinct("user_id").alias("n_distinct_user_id"),
    F.min("timestamp").alias("min_timestamp"),
    F.max("timestamp").alias("max_timestamp")
)

reviews_profile = reviews_profile.toPandas().T
reviews_profile.columns = ["value"]
display(reviews_profile)

,value
min_rating,1.000000
max_rating,5.000000
avg_rating,4.102635
min_helpful_vote,-4.000000
max_helpful_vote,11931.000000
n_distinct_asin,5318892.000000
n_distinct_parent_asin,4320211.000000
n_distinct_user_id,22789619.000000
min_timestamp,885229108000.000000
max_timestamp,1694622623621.000000


It's weird to have negative helpful_votes. Timestamp is formatted in milliseconds

In [24]:
missing = [
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1)).alias(c)
    for c in df_reviews.columns
]
missing_review = df_reviews.agg(*missing)

missing_review = missing_review.toPandas().T
missing_review.columns = ["missing count"]
display(missing_review)

,missing count
asin,0
helpful_vote,0
parent_asin,0
rating,0
text,88026
timestamp,0
title,0
user_id,0
verified_purchase,0


There are 88026 rows where text is missing/empty. Blank review texts are not necessary for further investigation.

In [27]:
## remove empty/null text & negative helpful votes
reviews_clean = (
    df_reviews
    .filter(F.col("text").isNotNull())
    .filter(F.length(F.trim(F.col("text"))) > 0)
    
    .filter(F.col("helpful_vote") >= 0)
)

before = df_reviews.count()
after = reviews_clean.count()

print("Before:", before)
print("After:", after)
print("Rows removed:", before - after)

Before: 64679785
After: 64591753
Rows removed: 88032


In [30]:
## convert timestamp
reviews_clean = reviews_clean.withColumn(
    "timestamp_df",
    (F.col("timestamp") / 1000).cast("timestamp")
)
reviews_clean = reviews_clean.drop("timestamp")

In [31]:
reviews_clean.show(2)

+----------+------------+-----------+------+--------------------+--------------------+--------------------+-----------------+--------------------+
|      asin|helpful_vote|parent_asin|rating|                text|               title|             user_id|verified_purchase|        timestamp_df|
+----------+------------+-----------+------+--------------------+--------------------+--------------------+-----------------+--------------------+
|B00HSAOFD8|           0| B00HSAOFD8|   4.0|Brushes arrived q...|          Four Stars|AGWL43EG6H3AO3CN2...|             true| 2016-01-26 14:40:07|
|B07MGKRMBG|           0| B07SPY1GJG|   5.0|I have tried just...|Finally a natural...|AGUUIUEN3Y4X54WXM...|             true|2020-09-17 11:15:...|
+----------+------------+-----------+------+--------------------+--------------------+--------------------+-----------------+--------------------+
only showing top 2 rows


In [33]:
# save cleaned reviews data
output_path = "gs://msca-bdp-students-bucket/notebooks/sharonlee/Final Project/review_data"
reviews_clean.write \
    .mode("overwrite") \
    .parquet(output_path)

print("Saved cleaned data to:", output_path)

Saved cleaned data to: gs://msca-bdp-students-bucket/notebooks/sharonlee/Final Project/review_data


#### Meta

In [53]:
df_meta.limit(2)

author,average_rating,bought_together,categories,description,main_category,parent_asin,price,rating_number,store,subtitle,title
NULL,5.0,NULL,[Beauty & Persona...,[],All Beauty,B0771WJBHK,NULL,5,Bare Alchemy,NULL,Soothe Serum for ...
NULL,4.0,NULL,[Beauty & Persona...,[],NULL,B09NKNNS57,NULL,17,DISCOVER DEVICE,NULL,DISCOVER Tattoo B...


In [37]:
display(df_meta.printSchema())

root
 |-- author: struct (nullable = true)
 |    |-- about: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- avatar: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- bought_together: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- main_category: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- price: string (nullable = true)
 |-- rating_number: long (nullable = true)
 |-- store: string (nullable = true)
 |-- subtitle: string (nullable = true)
 |-- title: string (nullable = true)


None

In [38]:
meta_profile = df_meta.agg(
    F.min("average_rating").alias("min_average_rating"),
    F.max("average_rating").alias("max_average_rating"),
    F.avg("average_rating").alias("avg_average_rating"),

    F.min("rating_number").alias("min_rating_number"),
    F.max("rating_number").alias("max_rating_number"),
    F.avg("rating_number").alias("avg_rating_number"),

    F.countDistinct("parent_asin").alias("n_distinct_parent_asin"),
    F.countDistinct("title").alias("n_distinct_title"),
    F.countDistinct("main_category").alias("n_distinct_main_category"),
    F.countDistinct("store").alias("n_distinct_store"),

    F.count(F.when(~F.col("price").rlike("^[0-9\\.,$]+$"), True)).alias("n_weird_price_format")
)

meta_profile = meta_profile.toPandas().T
meta_profile.columns = ["value"]

display(meta_profile)

,value
min_average_rating,1.000000
max_average_rating,5.000000
avg_average_rating,4.074292
min_rating_number,1.000000
max_rating_number,354024.000000
avg_rating_number,119.660846
n_distinct_parent_asin,4320533.000000
n_distinct_title,4178415.000000
n_distinct_main_category,42.000000
n_distinct_store,295565.000000


In [45]:
weird_price_condition = (
    F.col("price").isNotNull() &
    ~F.col("price").rlike(r"^[0-9\.,$]+$")
)

df_meta.filter(weird_price_condition) \
       .select("price") \
       .distinct() \
       .show(10, truncate=False)

+----------+
|price     |
+----------+
|from 49.50|
|from 32.95|
|from 9.00 |
|—         |
|from 39.98|
|from 23.99|
|from 20.00|
|from 12.98|
|from 15.99|
|from 22.99|
+----------+
only showing top 10 rows


2029 rows have a price value that doesn't look like a simple money string.  
Possible formats: "from 49.50" "Unavailable", "N/A"/"null" as a string, None, etc.
They are not NULL.

In [50]:
# top-level missing counts
top_missing = df_meta.agg(*[
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in df_meta.columns
])

top_missing = top_missing.toPandas().T
top_missing.columns = ["missing_count"]
display(top_missing)

,missing_count
author,4320377
average_rating,0
bought_together,4320533
categories,0
description,0
main_category,249444
parent_asin,0
price,2555695
rating_number,0
store,87815


In [51]:
nested_missing = df_meta.agg(
    F.count(F.when(F.col("author").isNull(), 1)).alias("author_null"),
    F.count(F.when(F.col("author.name").isNull(), 1)).alias("author_name_null"),
    F.count(F.when(F.col("author.avatar").isNull(), 1)).alias("author_avatar_null"),
    F.count(F.when(F.col("author.about").isNull(), 1)).alias("author_about_null")
)

nested_missing = nested_missing.toPandas().T
nested_missing.columns = ["missing_count"]
display(nested_missing)

,missing_count
author_null,4320377
author_name_null,4320377
author_avatar_null,4320377
author_about_null,4320377


In [52]:
arrays_missing = df_meta.agg(
    F.count(F.when(F.col("categories").isNull(), 1)).alias("categories_null"),
    F.count(F.when(F.size("categories") == 0, 1)).alias("categories_empty"),
    F.count(F.when(F.col("description").isNull(), 1)).alias("description_null"),
    F.count(F.when(F.size("description") == 0, 1)).alias("description_empty")
)

arrays_missing = arrays_missing.toPandas().T
arrays_missing.columns = ["count"]
display(arrays_missing)

,count
categories_null,0
categories_empty,355136
description_null,0
description_empty,1855488


There are some columns that contain lots of NULL and also are not relevant to our project.  
So drop irrelevant columns, and clean some of "price" values.  

In [54]:
# remove irrelevant columns
meta_clean = (
    df_meta
    .select(
        "parent_asin",
        "title",
        "main_category",
        "price",
        "average_rating",
        "rating_number"
    )
    # add cleaned numeric price
    .withColumn(
        "price_num",
        F.regexp_extract(F.col("price"), r"([0-9]+(?:[\\.,][0-9]+)?)", 1)
    )
    .withColumn(
        "price_num",
        F.when(F.col("price_num") == "", None).otherwise(F.col("price_num")).cast("double")
    )
)

# just extract numeric price values. if it doesn't contain any numeric values then make it NULL

In [55]:
meta_clean.limit(5)

parent_asin,title,main_category,price,average_rating,rating_number,price_num
B0771WJBHK,Soothe Serum for ...,All Beauty,NULL,5.0,5,NULL
B09NKNNS57,DISCOVER Tattoo B...,NULL,NULL,4.0,17,NULL
B0C6FF5HWL,auroray 2200 Pcs ...,All Beauty,8.99,4.6,137,8.99
B09NKSHV3Z,KKNC Chemistry In...,NULL,15.69,1.0,2,15.69
B000C213AC,Aficionado By Fin...,All Beauty,NULL,5.0,1,NULL


In [56]:
# save cleaned meta data
output_path1 = "gs://msca-bdp-students-bucket/notebooks/sharonlee/Final Project/meta_data"
meta_clean.write \
    .mode("overwrite") \
    .parquet(output_path1)

print("Saved cleaned data to:", output_path1)

Saved cleaned data to: gs://msca-bdp-students-bucket/notebooks/sharonlee/Final Project/meta_data
